# 10 - Operational Prioritization

## Objective

This notebook transforms model predictions into operational recommendations by scoring the September–October validation window, ranking high-risk flights, applying capacity-constrained prioritization logic, and evaluating prioritized selection against a random baseline for RQ4/H4.

Outputs support the Streamlit prioritization tab, the final report, and downstream dashboard preparation.

#### Load project configuration

In [0]:
from config import project_config as cfg

print("Project configuration loaded successfully.")
print(f"Predictions table: {cfg.PREDICTIONS_TABLE}")
print(f"Prioritization table: {cfg.PRIORITIZATION_RESULTS_TABLE}")
print(f"Scoring window: {cfg.SCORING_START_DATE} to {cfg.SCORING_END_DATE}")


#### Load the saved model and modeling checkpoints

The selected Spark ML model and the validation-period modeling datasets produced in notebooks 07 and 08 are loaded before batch scoring.

In [0]:
from __future__ import annotations

import json

from pyspark.ml.classification import LogisticRegressionModel
from pyspark.sql import functions as F
from utils.model_training import (
    create_feature_hasher_from_manifest,
    hash_modeling_frame,
    load_hist_modeling_table,
)


def require_table(table_name: str) -> None:
    if not spark.catalog.tableExists(table_name):
        raise RuntimeError(
            f"Required table '{table_name}' was not found. "
            "Run notebooks 07, 08, and 09 before continuing."
        )


def require_file(file_path: str) -> None:
    try:
        dbutils.fs.head(file_path, 1)
    except Exception as exc:
        raise RuntimeError(
            f"Required file '{file_path}' was not found."
        ) from exc

def require_model_path(model_path: str) -> None:
    try:
        dbutils.fs.ls(model_path)
    except Exception as exc:
        raise RuntimeError(
            f"Required model path '{model_path}' was not found. "
            "Run notebook 08 through the model-save cells."
        ) from exc

require_model_path(cfg.SELECTED_MODEL_PATH)
require_file(cfg.SELECTED_MODEL_METRICS_PATH)
require_file(cfg.MODEL_FEATURE_MANIFEST_PATH)

for table_name in [
    cfg.MODELING_VALIDATION_HIST_TABLE,
    cfg.SHAP_GLOBAL_IMPORTANCE_TABLE,
]:
    require_table(table_name)

feature_manifest = json.loads(
    dbutils.fs.head(cfg.MODEL_FEATURE_MANIFEST_PATH, 1000000)
)
model_metrics = json.loads(
    dbutils.fs.head(cfg.SELECTED_MODEL_METRICS_PATH, 1000000)
)

TARGET_COLUMN = feature_manifest["target_column"]
DECISION_THRESHOLD = float(
    model_metrics.get(
        "selected_validation_threshold",
        cfg.DEFAULT_DECISION_THRESHOLD,
    )
)

final_model = LogisticRegressionModel.load(cfg.SELECTED_MODEL_PATH)
df_validation_hist = load_hist_modeling_table(cfg.MODELING_VALIDATION_HIST_TABLE)
feature_hasher = create_feature_hasher_from_manifest(feature_manifest)
df_validation_hashed = hash_modeling_frame(
    df_validation_hist,
    feature_hasher,
)
global_importance_pdf = (
    spark.table(cfg.SHAP_GLOBAL_IMPORTANCE_TABLE)
    .orderBy(F.col("MeanAbsSHAP").desc())
    .toPandas()
)
top_shap_feature = global_importance_pdf.iloc[0]["Feature"]

print("Saved model and checkpoints loaded successfully.")
print(f"Decision threshold: {DECISION_THRESHOLD:.2f}")
print(f"Model input columns: {len(feature_manifest['model_input_columns'])}")
print(f"Top global SHAP driver: {top_shap_feature}")


#### Score the operational review window

The September–October validation window is used as the operational scoring period because it contains completed flights with known delay outcomes and supports RQ4 evaluation without using the final holdout test set.

In [0]:
from pyspark.ml.functions import vector_to_array

scored_predictions = final_model.transform(df_validation_hashed)

scored_predictions = (
    scored_predictions
    .withColumn(
        "delay_probability",
        F.element_at(
            vector_to_array(F.col("probability")),
            2,
        ),
    )
    .withColumn(
        "predicted_delay",
        (
            F.col("delay_probability") >= F.lit(DECISION_THRESHOLD)
        ).cast("double"),
    )
    .select(
        *cfg.MODELING_JOIN_KEY_COLUMNS,
        F.col(TARGET_COLUMN).alias("actual_delay"),
        "predicted_delay",
        "delay_probability",
    )
)

display(scored_predictions.limit(5))
print(f"Scored flights: {scored_predictions.count():,}")
print(f"Operational decision threshold: {DECISION_THRESHOLD:.2f}")

In [0]:
import pandas as pd

from utils.operational_prioritization import (
    add_operational_scores,
    assign_shap_main_drivers,
)


prediction_attributes = df_validation_hist.select(
    *cfg.MODELING_JOIN_KEY_COLUMNS,
    cfg.MONTH_COLUMN,
    cfg.TIME_OF_DAY_COLUMN,
    cfg.SEASON_COLUMN,
)

predictions_frame = (
    scored_predictions.join(
        prediction_attributes,
        on=cfg.MODELING_JOIN_KEY_COLUMNS,
        how="inner",
    )
    .withColumn(
        "flight_label",
        F.concat_ws(
            "-",
            F.col(cfg.AIRLINE_COLUMN),
            F.col(cfg.FLIGHT_NUMBER_COLUMN).cast("string"),
            F.col(cfg.ORIGIN_COLUMN),
            F.col(cfg.DESTINATION_COLUMN),
            F.col(cfg.SCHEDULED_DEPARTURE_COLUMN).cast("string"),
        ),
    )
    .withColumn(
        "scheduled_departure_text",
        F.date_format(
            F.to_timestamp(
                F.lpad(F.col(cfg.SCHEDULED_DEPARTURE_COLUMN).cast("string"), 4, "0"),
                "HHmm",
            ),
            "HH:mm",
        ),
    )
)

scored_count = scored_predictions.count()
joined_count = predictions_frame.count()
if joined_count != scored_count:
    raise ValueError(
        "Prediction join inflated or dropped rows: "
        f"scored={scored_count:,}, joined={joined_count:,}. "
        "Re-run notebooks 06 and 07 after syncing the flight-number join key."
    )

predictions_pdf = predictions_frame.toPandas()
predictions_pdf = predictions_pdf.rename(
    columns={
        cfg.AIRLINE_COLUMN: "airline_code",
        cfg.FLIGHT_NUMBER_COLUMN: "flight_number",
        cfg.ORIGIN_COLUMN: "origin_airport",
        cfg.DESTINATION_COLUMN: "destination_airport",
        cfg.SCHEDULED_DEPARTURE_COLUMN: "scheduled_departure",
        cfg.MONTH_COLUMN: "month_number",
        cfg.TIME_OF_DAY_COLUMN: "departure_window",
        cfg.SEASON_COLUMN: "season",
    }
)
predictions_pdf["shap_main_driver"] = assign_shap_main_drivers(
    predictions_pdf,
    global_importance_pdf,
    fallback_feature=top_shap_feature,
)
predictions_pdf = add_operational_scores(
    predictions_pdf,
    high_threshold=cfg.HIGH_RISK_THRESHOLD,
    critical_threshold=cfg.CRITICAL_RISK_THRESHOLD,
    medium_threshold=cfg.MEDIUM_RISK_THRESHOLD,
)

display(spark.createDataFrame(predictions_pdf).limit(5))
print(f"Operational predictions prepared: {len(predictions_pdf):,}")
print(f"Join check passed: {joined_count:,} scored flights retained.")


In [0]:
from utils.operational_prioritization import (
    build_ranking_table,
    compare_prioritization_strategies,
)


prioritization_pool = predictions_pdf[
    predictions_pdf["delay_probability"] >= cfg.PRIORITIZATION_POOL_MIN_PROB
].copy()

ranking_tables = []
evaluation_tables = []

for capacity_k in cfg.CAPACITY_K_OPTIONS:
    ranking = build_ranking_table(
        prioritization_pool,
        capacity_k=capacity_k,
        airline_column="airline_code",
        origin_column="origin_airport",
    )
    ranking["capacity_k"] = capacity_k
    ranking_tables.append(ranking)

    evaluation = compare_prioritization_strategies(
        prioritization_pool,
        capacity_k=capacity_k,
        random_seed=cfg.RANDOM_SEED,
        label_column="actual_delay",
        airline_column="airline_code",
        origin_column="origin_airport",
    )
    evaluation_tables.append(evaluation)

prioritization_results_pdf = pd.concat(ranking_tables, ignore_index=True)
prioritization_evaluation_pdf = pd.concat(evaluation_tables, ignore_index=True)

display(
    spark.createDataFrame(
        prioritization_evaluation_pdf[
            prioritization_evaluation_pdf["capacity_k"] == cfg.DEFAULT_CAPACITY_K
        ]
    )
)
display(
    spark.createDataFrame(
        prioritization_results_pdf[
            (prioritization_results_pdf["capacity_k"] == cfg.DEFAULT_CAPACITY_K)
            & (prioritization_results_pdf["selected"])
        ].head(20)
    )
)


#### Validate RQ4 / H4

RQ4 is supported when prioritized selection captures more delayed flights than a random baseline at the same operational capacity K. The comparison below provides the statistical evidence for the final report.

In [0]:
rq4_summary = prioritization_evaluation_pdf.copy()


random_results = (
    rq4_summary[
        rq4_summary["strategy"] == "Random Baseline"
    ][
        [
            "capacity_k",
            "captured_delayed_flights",
        ]
    ]
    .rename(
        columns={
            "captured_delayed_flights":
            "random_captured_delayed_flights"
        }
    )
)

rq4_summary = rq4_summary.merge(
    random_results,
    on="capacity_k",
    how="left",
)


rq4_summary["rq4_supported"] = (
    (
        rq4_summary["strategy"]
        == "Constrained Optimized Selection"
    )
    & (
        rq4_summary["captured_delayed_flights"]
        > rq4_summary["random_captured_delayed_flights"]
    )
)

display(
    spark.createDataFrame(
        rq4_summary
    )
)


default_k_results = rq4_summary[
    rq4_summary["capacity_k"]
    == cfg.DEFAULT_CAPACITY_K
].copy()

optimized_results = default_k_results[
    default_k_results["strategy"]
    == "Constrained Optimized Selection"
]

top_k_results = default_k_results[
    default_k_results["strategy"]
    == "Top-K Probability Baseline"
]

random_results_default = default_k_results[
    default_k_results["strategy"]
    == "Random Baseline"
]


if optimized_results.empty:
    raise ValueError(
        "Constrained Optimized Selection result was not found "
        f"for capacity K={cfg.DEFAULT_CAPACITY_K}."
    )

if top_k_results.empty:
    raise ValueError(
        "Top-K Probability Baseline result was not found "
        f"for capacity K={cfg.DEFAULT_CAPACITY_K}."
    )

if random_results_default.empty:
    raise ValueError(
        "Random Baseline result was not found "
        f"for capacity K={cfg.DEFAULT_CAPACITY_K}."
    )

optimized_row = optimized_results.iloc[0]
top_k_row = top_k_results.iloc[0]
random_row = random_results_default.iloc[0]


print(
    f"RQ4 default-K comparison at "
    f"K={cfg.DEFAULT_CAPACITY_K}:"
)

print(
    "Constrained optimized selection captured "
    f"{int(optimized_row['captured_delayed_flights'])} delayed flights "
    f"with precision "
    f"{optimized_row['delay_precision']:.2%}."
)

print(
    "Top-K probability baseline captured "
    f"{int(top_k_row['captured_delayed_flights'])} delayed flights "
    f"with precision "
    f"{top_k_row['delay_precision']:.2%}."
)

print(
    "Random baseline captured "
    f"{int(random_row['captured_delayed_flights'])} delayed flights "
    f"with precision "
    f"{random_row['delay_precision']:.2%}."
)

rq4_supported_default = (
    optimized_row["captured_delayed_flights"]
    > random_row["captured_delayed_flights"]
)

print(
    f"RQ4 supported at default capacity: "
    f"{rq4_supported_default}"
)

At higher capacity levels, the constrained optimization strategy may select fewer than \(K\) flights when airline and origin-airport diversification limits prevent additional feasible selections. For example, at \(K=100\), the optimizer selected 56 flights. Therefore, comparisons at this capacity should be interpreted with consideration of the binding diversification constraints.

#### Save operational outputs

In [0]:
predictions_df = spark.createDataFrame(predictions_pdf)
prioritization_results_df = spark.createDataFrame(prioritization_results_pdf)
prioritization_evaluation_df = spark.createDataFrame(prioritization_evaluation_pdf)

(
    predictions_df.write.format("delta").mode("overwrite").save(cfg.PREDICTIONS_DELTA_PATH)
)
(
    predictions_df.writeTo(cfg.PREDICTIONS_TABLE).using("delta").createOrReplace()
)

(
    prioritization_results_df.write.format("delta").mode("overwrite").save(
        cfg.PRIORITIZATION_RESULTS_PATH
    )
)
(
    prioritization_results_df.writeTo(cfg.PRIORITIZATION_RESULTS_TABLE)
    .using("delta")
    .createOrReplace()
)

(
    prioritization_evaluation_df.write.format("delta").mode("overwrite").save(
        cfg.PRIORITIZATION_EVALUATION_PATH
    )
)
(
    prioritization_evaluation_df.writeTo(cfg.PRIORITIZATION_EVALUATION_TABLE)
    .using("delta")
    .createOrReplace()
)

print("Operational prioritization outputs saved successfully.")
print(f"Predictions table: {cfg.PREDICTIONS_TABLE}")
print(f"Prioritization results: {cfg.PRIORITIZATION_RESULTS_TABLE}")
print(f"Prioritization evaluation: {cfg.PRIORITIZATION_EVALUATION_TABLE}")
